# 02 — Build Classifier

This notebook demonstrates loading data, generating embeddings, and running the classifier.

In [ ]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.preprocessing import preprocess_batch
from src.intent_classifier import IntentClassifier
from src.config import INTENTS


## Load Training Data

In [ ]:
train = pd.read_csv('../data/processed/train.csv')
val   = pd.read_csv('../data/processed/val.csv')
print(f"Training examples: {len(train)}")
print(f"Validation examples: {len(val)}")
train.head()


## Preprocess Text

In [ ]:
train['processed'] = preprocess_batch(train['message'].tolist())
val['processed']   = preprocess_batch(val['message'].tolist())
print("Sample preprocessed messages:")
for i in range(3):
    print(f"  Raw:  {train['message'].iloc[i]}")
    print(f"  Proc: {train['processed'].iloc[i]}")
    print()


## Train the Classifier

This encodes all training examples using `all-MiniLM-L6-v2`.

In [ ]:
clf = IntentClassifier(k=5)
clf.fit(train['processed'].tolist(), train['intent'].tolist(), show_progress=True)
print("Classifier fitted!")
print(f"Training embeddings shape: {clf._train_embeddings.shape}")


## Predictions on Validation Set

In [ ]:
results = clf.predict_batch(val['processed'].tolist())
val['predicted_intent'] = [r['intent'] for r in results]
val['confidence'] = [r['confidence'] for r in results]

from sklearn.metrics import accuracy_score, classification_report
acc = accuracy_score(val['intent'], val['predicted_intent'])
print(f"Validation Accuracy: {acc:.4f}")
print()
print(classification_report(val['intent'], val['predicted_intent'], zero_division=0))


## Sample Predictions with Confidence

In [ ]:
sample_msgs = [
    "my refund still hasn't arrived",
    "app keeps crashing every time I open it",
    "I forgot my password and can't login",
    "Love your service, thank you!",
    "Could you please add dark mode?",
    "This is RIDICULOUS! I was charged twice!!!",
]

print(f"{'Message':<50} {'Intent':<20} {'Confidence'}")
print("-" * 80)
for msg in sample_msgs:
    from src.preprocessing import preprocess
    r = clf.predict_with_confidence(preprocess(msg))
    print(f"{msg[:48]:<50} {r['intent']:<20} {r['confidence']:.4f}")


## Confidence Distribution

In [ ]:
import matplotlib.pyplot as plt
correct = val['intent'] == val['predicted_intent']
plt.figure(figsize=(10, 5))
plt.hist(val[correct]['confidence'], bins=30, alpha=0.6, label='Correct', color='steelblue')
plt.hist(val[~correct]['confidence'], bins=30, alpha=0.6, label='Incorrect', color='salmon')
plt.axvline(x=0.60, color='red', linestyle='--', label='Threshold (0.60)')
plt.xlabel('Confidence (Mean Cosine Similarity)')
plt.ylabel('Count')
plt.title('Confidence Distribution: Correct vs Incorrect Predictions')
plt.legend()
plt.tight_layout()
plt.show()
